# Analysis: probe, statistics, tables and figures

Everything here reads the JSON/JSONL artefacts written during training, so it needs **no GPU** (except the frozen-feature probe, which is cheap). You can run the same commands on your laptop to iterate on figures.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command, streaming its output live.

    Streaming rather than capture_output matters because these jobs run for
    tens of minutes and report progress as they go. Buffering that until the
    process exits makes a long job indistinguishable from a hung one.
    """
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: a child process writing to a pipe switches from line
    # buffering to 4 KB block buffering, so progress lines would still arrive
    # in bursts (or not at all until exit) even though we stream them here.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code_ = p.wait()
    if check and code_ != 0:
        # Include the tail of the output in the exception. Otherwise the
        # traceback shows only this wrapper and the real error is buried
        # further up the cell, which is easy to miss and impossible to
        # copy/paste usefully.
        tail = ''.join(lines[-25:]).rstrip()
        raise RuntimeError(
            f'command failed (exit {code_}): {cmd}\n\n--- last output ---\n{tail}')
    return subprocess.CompletedProcess(cmd, code_, ''.join(lines), '')

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Report the accelerator and the precision that follows from it.
# T4 (sm_75) has fp16 tensor cores but NO bf16 hardware; P100 (sm_60) has
# neither and runs ~2x slower. The code adapts either way — this cell is
# here so you know what you were given before spending 8 hours on it.
import torch
from src.config import amp_config
print('CUDA devices:', torch.cuda.device_count())
amp = amp_config()
print(amp)
if torch.cuda.device_count() < 2:
    print('\n*** Only one GPU. I-JEPA and MAE will still run correctly, but\n'
          '    SimCLR/MoCo v3 need 2 GPUs to preserve global_batch=512.\n'
          '    Set Session options -> Accelerator -> GPU T4 x2 and re-run. ***')


In [ ]:
# k-NN + linear probe on frozen features. The cheapest signal about
# representation quality — run it before trusting any segmentation number.
sh('python -m src.eval.probe --save-embeddings', check=False)


In [ ]:
sh('python -m src.eval.stats')


In [ ]:
sh('python -m src.eval.tables')


In [ ]:
sh('python -m src.viz.make_all --out /kaggle/working/figures')


In [ ]:
from IPython.display import Image as IPyImage, display, Markdown
import glob
for f in sorted(glob.glob('/kaggle/working/figures/*.png')):
    display(Markdown(f'### {os.path.basename(f)}'))
    display(IPyImage(filename=f))


In [ ]:
display(Markdown(open('/kaggle/working/results/tables/all_tables.md').read()))
